In [2]:
from pathlib import Path
from dotenv import load_dotenv
import re
import os
import smtplib
import pymongo
import warnings
import numpy as np
import pandas as pd
import datetime as dt
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt
from flatten_json import flatten
from google.cloud import bigquery
from bson.objectid import ObjectId
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
env_path = Path('/home/aurora') / '.env'
load_dotenv(dotenv_path=env_path)
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])

In [3]:
end = dt.datetime.today() - dt.timedelta(1)
end = end.replace(hour=18, minute=30, second=0, microsecond=0)
start = end - dt.timedelta(1)
print(start, end, end - start)

2019-06-24 18:30:00 2019-06-25 18:30:00 1 day, 0:00:00


In [4]:
aw = []
c_users = cursor.superstars.users
for documents in c_users.find({},{'app_data'}): 
    aw.append(documents)

dic_flattened = [flatten(d) for d in aw]
users = pd.DataFrame(dic_flattened)

In [5]:
users = users[['_id','app_data_0_app_kind']]
users = users.fillna(1)
tgpl_users = users[users['app_data_0_app_kind']=='2']
del users

In [29]:
print('Querying Collectables Logs Table')
c_ucl = cursor.superstars.user_collectables_logs
aw = []
for documents in c_ucl.aggregate([{'$sort' : {'_id' : -1}}, 
                                  {"$match" : {"type" : "HARD_CURRENCY"}}, 
                                    {"$match" : {"user" : ObjectId("5d0d1deafa80e3002820af05")}}, 
                                    {'$unwind':"$data"}]):
    aw.append(documents)
# for documents in c_ucl.find({"user":ObjectId("5d0d1deafa80e3002820af05")}): 
#     aw.append(documents)

dic_flattened = [flatten(d) for d in aw]
df_main = pd.DataFrame(dic_flattened)
df_main

Querying Collectables Logs Table


,__v,_id,data__id,data_action_item_id,data_action_item_type,data_catalogue_id,data_created_at,data_quantity,data_reason_type,type,user
0,0,5d0d2028fa80e30028215807,5d0d2028fa80e30028215808,NaN,NaN,1,2019-06-21 18:21:28.845,5,LEVEL_UP,HARD_CURRENCY,5d0d1deafa80e3002820af05
1,0,5d0d2028fa80e30028215807,5d0d209dfa80e30028217374,NaN,NaN,1,2019-06-21 18:23:25.023,5,LEVEL_UP,HARD_CURRENCY,5d0d1deafa80e3002820af05
2,0,5d0d2028fa80e30028215807,5d0d278e8185c200191e845f,NaN,NaN,1,2019-06-21 18:53:02.011,5,LEVEL_UP,HARD_CURRENCY,5d0d1deafa80e3002820af05
3,0,5d0d2028fa80e30028215807,5d0d2a25fa80e3002822abc9,NaN,NaN,1,2019-06-21 19:04:05.005,-1,QUICK_TRAINING,HARD_CURRENCY,5d0d1deafa80e3002820af05
4,0,5d0d2028fa80e30028215807,5d0d3197fa80e30028235a25,NaN,NaN,1,2019-06-21 19:35:51.293,-1,END_TRAINING,HARD_CURRENCY,5d0d1deafa80e3002820af05
5,0,5d0d2028fa80e30028215807,5d0d31a08185c200191f510e,NaN,NaN,1,2019-06-21 19:36:00.567,-2,QUICK_TRAINING,HARD_CURRENCY,5d0d1deafa80e3002820af05
6,0,5d0d2028fa80e30028215807,5d0d31aefa80e30028236606,NaN,NaN,1,2019-06-21 19:36:14.543,-1,QUICK_TRAINING,HARD_CURRENCY,5d0d1deafa80e3002820af05
7,0,5d0d2028fa80e30028215807,5d0d31ba8185c200191f51df,NaN,NaN,1,2019-06-21 19:36:26.086,-2,QUICK_TRAINING,HARD_CURRENCY,5d0d1deafa80e3002820af05
8,0,5d0d2028fa80e30028215807,5d0d31d4fa80e30028237106,NaN,NaN,1,2019-06-21 19:36:52.778,-1,QUICK_TRAINING,HARD_CURRENCY,5d0d1deafa80e3002820af05
9,0,5d0d2028fa80e30028215807,5d0d31e6fa80e30028237175,NaN,NaN,1,2019-06-21 19:37:10.108,-2,QUICK_TRAINING,HARD_CURRENCY,5d0d1deafa80e3002820af05


In [32]:
df_main.groupby('data_reason_type').agg({'data_quantity':'sum'})

,data_quantity
data_reason_type,
CLAIMED_FROM_MAIL,21
DAILY_KIT_REFRESH,-215
DAILY_KIT_REWARDS,-668
END_TRAINING,-60
ENERGY_PURCHASE,-370
KIT_KEYS_PURCHASE,-25650
LEVEL_UP,125
QUICK_TRAINING,-1067
SPEEDUP_USE,-10


In [36]:
aw = []
c_users = cursor.superstars.users
for documents in c_users.find({"_id":ObjectId("5d0d1deafa80e3002820af05")}): 
    aw.append(documents)


dic_flattened = [flatten(d) for d in aw]
users = pd.DataFrame(dic_flattened)
aw

[{'_id': ObjectId('5d0d1deafa80e3002820af05'),
  'collectables': {'player_items': {'upgrades': {'training_cards': [{'quantity': 3,
       '_id': ObjectId('5d133a79114e23001a80088e'),
       'id': 2},
      {'quantity': 3, '_id': ObjectId('5d133a79114e23001a80088d'), 'id': 3},
      {'quantity': 26, '_id': ObjectId('5d133a627119e600139c838b'), 'id': 4},
      {'quantity': 1, '_id': ObjectId('5d133a79114e23001a80088c'), 'id': 45},
      {'quantity': 1, '_id': ObjectId('5d133a79114e23001a80088b'), 'id': 46}],
     'training_speedup_cards': [{'quantity': 0,
       '_id': ObjectId('5d12ff12c5ebf80011a4e957'),
       'id': 1},
      {'quantity': 1, '_id': ObjectId('5d12ff12c5ebf80011a4e956'), 'id': 2},
      {'quantity': 1, '_id': ObjectId('5d12ff12c5ebf80011a4e955'), 'id': 3},
      {'quantity': 173, '_id': ObjectId('5d12ff12c5ebf80011a4e954'), 'id': 4},
      {'quantity': 17, '_id': ObjectId('5d126ff972353600186c7f80'), 'id': 5}],
     'passive_tokens': [],
     'active_tokens': [],
     '

In [9]:
temp = df_main[df_main['user']=='5d0d1deafa80e3002820af05']
temp

,__v,_id,data__id,data_action_item_id,data_action_item_type,data_catalogue_id,data_created_at,data_quantity,data_reason_type,type,user


In [7]:
df = df_main[~df_main['user'].isin(tgpl_users['_id'])]

print('Data Wrangling')
hitcoins = df[df['data_catalogue_id']== '1']

hitcoins = hitcoins[ hitcoins['data_quantity'] < 0 ]
hitcoins['data_quantity'] = -hitcoins['data_quantity']

_ = hitcoins.groupby(['data_reason_type','user']).agg({'data_quantity':'sum'}).reset_index()
_

Data Wrangling


,data_reason_type,user,data_quantity
0,DAILY_KIT_REFRESH,5c97b32e9732f62d3e3adb36,20
1,DAILY_KIT_REFRESH,5cbe8f7b2bdfca07d876db65,5
2,DAILY_KIT_REFRESH,5cd802e6822ea36f394009c2,5
3,DAILY_KIT_REFRESH,5cf1312767847c1b7cf3b1af,5
4,DAILY_KIT_REFRESH,5cf67555f588234bca10205d,5
5,DAILY_KIT_REFRESH,5d03be72ad6fe1001851653c,5
6,DAILY_KIT_REFRESH,5d06a9b7a6264f03597bcbac,5
7,DAILY_KIT_REFRESH,5d07107cc61219034a416293,5
8,DAILY_KIT_REFRESH,5d07c55fa6264f0359a380da,5
9,DAILY_KIT_REFRESH,5d0d1deafa80e3002820af05,70


In [ ]:
_ = _[_['data']]

In [18]:
c_payment_orders = cursor.superstars.payment_orders

aw = []
for documents in c_payment_orders.find({"updated_at" : {"$gte" : start, "$lte" : end}}  ):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
purchased = pd.DataFrame(dic_flattened)
purchased

,__v,_id,created_at,currency_code,details_gateway,details_order_id,details_returned_params,items_0__id,items_0_id,items_0_type,revenue_bottom_line,revenue_top_line,status,type,updated_at,user_email,user_id,user_name
0,0,5d0a9d313c8b7e0018d40161,2019-06-19 20:38:09.131,INR,google_in_app_billing,GPA.3359-5045-6146-82064,"{""receipt"":""{\""Store\"":\""GooglePlay\"",\""Transa...",5d0a9d313c8b7e0018d40162,6,HARD_CURRENCY_PACKAGE,5530,7900,2,GENERAL_PURCHASE,2019-06-19 20:38:10.361,hw+1560975497461@hws.com,5d0a9889d0dff00011543f46,Flintstone


In [ ]:
df = df_main[~df_main['user'].isin(tgpl_users['_id'])]

print('Data Wrangling')
hitcoins = df[df['data_catalogue_id']== '1']

hitcoins = hitcoins[ hitcoins['data_quantity'] < 0 ]
hitcoins['data_quantity'] = -hitcoins['data_quantity']

_ = hitcoins.groupby(['data_reason_type','user']).agg({'data_quantity':'sum'}).reset_index()
_ = _.groupby(['data_reason_type']).agg({'data_quantity':'max'}).reset_index()

hitcoins = hitcoins.groupby(['data_reason_type']).agg({'data_quantity':'sum','user':pd.Series.nunique}).reset_index()
hitcoins['Hitcoins Spent per User'] = (hitcoins['data_quantity'] / hitcoins['user']).round(1)

hitcoins = pd.merge(hitcoins,_,on='data_reason_type',how='inner')
hitcoins.columns = ['Feature','Total HC Spent','Unique Users','HC Spent per Unique User','Max HC Spent']
hitcoins['Feature'] = hitcoins['Feature'].replace({"DAILY_KIT_REFRESH" : "Daily Deals Refresh",
                                                     "DAILY_KIT_REWARDS" : "Daily Deals Purchase",
                                                     "END_TRAINING" : "Finish Training",
                                                     "ENERGY_PURCHASE" : "Energy Purchase",
                                                     "KIT_KEYS_PURCHASE" : "Sponsor Box Open",
                                                     "QUICK_TRAINING" : "Instant Training",
                                                     "SPEEDUP_USE" : "Speedup Purchase"})

custom_dict = {"Instant Training":0,"Finish Training":1,"Sponsor Box Open":2,"Speedup Purchase":3,
               "Daily Deals Purchase":4,"Daily Deals Refresh":5,"Energy Purchase":6}
hitcoins = hitcoins.iloc[hitcoins['Feature'].map(custom_dict).argsort()]
ss_total = hitcoins['Total HC Spent'].sum()

hitcoins['Total HC Spent'] = hitcoins.apply(lambda x: "{:,}".format(x['Total HC Spent']), axis=1)
hitcoins['Unique Users'] = hitcoins.apply(lambda x: "{:,}".format(x['Unique Users']), axis=1)
hitcoins['HC Spent per Unique User'] = hitcoins.apply(lambda x: "{:,}".format(x['HC Spent per Unique User']), axis=1)
hitcoins['Max HC Spent'] = hitcoins.apply(lambda x: "{:,}".format(x['Max HC Spent']), axis=1)
ss_hitcoins = hitcoins
    
    try:
        df = df_main[df_main['user'].isin(tgpl_users['_id'])]

        hitcoins = df[df['data_catalogue_id']== '1']

        hitcoins = hitcoins[ hitcoins['data_quantity'] < 0 ]
        hitcoins['data_quantity'] = -hitcoins['data_quantity']

        _ = hitcoins.groupby(['data_reason_type','user']).agg({'data_quantity':'sum'}).reset_index()
        _ = _.groupby(['data_reason_type']).agg({'data_quantity':'max'}).reset_index()

        hitcoins = hitcoins.groupby(['data_reason_type']).agg({'data_quantity':'sum','user':pd.Series.nunique}).reset_index()
        hitcoins['Hitcoins Spent per User'] = (hitcoins['data_quantity'] / hitcoins['user']).round(1)

        hitcoins = pd.merge(hitcoins,_,on='data_reason_type',how='inner')
        hitcoins.columns = ['Feature','Total HC Spent','Unique Users','HC Spent per Unique User','Max HC Spent']
        hitcoins['Feature'] = hitcoins['Feature'].replace({"DAILY_KIT_REFRESH" : "Daily Deals Refresh",
                                                             "DAILY_KIT_REWARDS" : "Daily Deals Purchase",
                                                             "END_TRAINING" : "Finish Training",
                                                             "ENERGY_PURCHASE" : "Energy Purchase",
                                                             "KIT_KEYS_PURCHASE" : "Sponsor Box Open",
                                                             "QUICK_TRAINING" : "Instant Training",
                                                             "SPEEDUP_USE" : "Speedup Purchase"})

        custom_dict = {"Instant Training":0,"Finish Training":1,"Sponsor Box Open":2,"Speedup Purchase":3,
                       "Daily Deals Purchase":4,"Daily Deals Refresh":5,"Energy Purchase":6}
        hitcoins = hitcoins.iloc[hitcoins['Feature'].map(custom_dict).argsort()]
        tgpl_total = hitcoins['Total HC Spent'].sum()


        hitcoins['Total HC Spent'] = hitcoins.apply(lambda x: "{:,}".format(x['Total HC Spent']), axis=1)
        hitcoins['Unique Users'] = hitcoins.apply(lambda x: "{:,}".format(x['Unique Users']), axis=1)
        hitcoins['HC Spent per Unique User'] = hitcoins.apply(lambda x: "{:,}".format(x['HC Spent per Unique User']), axis=1)
        hitcoins['Max HC Spent'] = hitcoins.apply(lambda x: "{:,}".format(x['Max HC Spent']), axis=1)
        tgpl_hitcoins = hitcoins
    except:
        data = [["Daily Deals Refresh",0,0,0,0],
                ["Daily Deals Purchase",0,0,0,0],
                ["Finish Training",0,0,0,0],
                ["Energy Purchase",0,0,0,0],
                ["Sponsor Box Open",0,0,0,0],
                ["Instant Training",0,0,0,0],
                ["Speedup Purchase",0,0,0,0]] 
        tgpl_hitcoins= pd.DataFrame(data, columns = ['Feature','Total HC Spent','Unique Users','HC Spent per Unique User','Max HC Spent']) 
        tgpl_hitcoins
    print('Getting Purchase Details')
    
    cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                                 os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                                 os.environ['dbname']) 
    c_payment_orders = cursor.superstars.payment_orders
    
    aw = []
    for documents in c_payment_orders.find({"updated_at" : {"$gte" : start, "$lte" : end}}  ):
        aw.append(documents)
    dic_flattened = [flatten(d) for d in aw]
    purchased = pd.DataFrame(dic_flattened)
    
    try:
            purchased = purchased[purchased['status']==2]
            purchased = purchased[purchased['currency_code']!="TGPL"]
            purchased = purchased[['items_0_id','items_0_id']]
            purchased.columns = ['id','type']

            data = [[1,50,49], [2,250,199], [3,700,499], [4,2500,1599], [5,6500,3999], [6,14000,7900]] 
            rates = pd.DataFrame(data, columns = ['id', 'hitcoins', 'rates']) 
            rates['hitcoins'] = rates.apply(lambda x: "{:,}".format(x['hitcoins']), axis=1)

            purchased = pd.merge (purchased,rates,on='id',how='inner')
            purchased = purchased['hitcoins'].astype(int).sum()
    except:
            purchased = 0
    ss_purchased=purchased
    
    aw = []
    for documents in c_payment_orders.find({"updated_at" : {"$gte" : start, "$lte" : end}}  ):
        aw.append(documents)
    dic_flattened = [flatten(d) for d in aw]
    purchased = pd.DataFrame(dic_flattened)

    try:
            purchased = purchased[purchased['status']==2]
            purchased = purchased[purchased['currency_code']=='TGPL']
            purchased = purchased[['items_0_id','items_0_id']]
            purchased.columns = ['id','type']

            data = [[1,50,49], [2,250,199], [3,700,499], [4,2500,1599], [5,6500,3999], [6,14000,7900]] 
            rates = pd.DataFrame(data, columns = ['id', 'hitcoins', 'rates']) 
            rates['hitcoins'] = rates.apply(lambda x: "{:,}".format(x['hitcoins']), axis=1)

            purchased = pd.merge (purchased,rates,on='id',how='inner')
            purchased = purchased['hitcoins'].astype(int).sum()
    except:
            purchased = 0
    tgpl_purchased=purchased

    numeric_col_mask = ss_hitcoins.dtypes.apply(lambda d: issubclass(np.dtype(d).type, np.number))
    
    # Dict used to center the table headers
    d = dict(selector="col_heading",
        props=[('text-align', 'center'),('font-weight', 'bold'),('font-size', '14px')])
    
    print('Styling')
    style1 = ss_hitcoins.style.set_properties(subset=ss_hitcoins.columns[numeric_col_mask],**{'width':'10em', 'text-align':'right'})\
            .set_properties(subset=ss_hitcoins.columns[~numeric_col_mask],**{'width':'10em', 'text-align':'left'})\
            .set_table_styles([d]).render()
    
    
    numeric_col_mask = tgpl_hitcoins.dtypes.apply(lambda d: issubclass(np.dtype(d).type, np.number))
    
    d = dict(selector="col_heading",
        props=[('text-align', 'center'),('font-weight', 'bold'),('font-size', '14px')])
    
    style2 = tgpl_hitcoins.style.set_properties(subset=tgpl_hitcoins.columns[numeric_col_mask],**{'width':'10em', 'text-align':'right'})\
            .set_properties(subset=tgpl_hitcoins.columns[~numeric_col_mask],**{'width':'10em', 'text-align':'left'})\
            .set_table_styles([d]).render()
    print('Preparing Mail Content')
    html_str = """<html>
    <head>
    <style>
    
        h2 {
            font-family: Helvetica, Arial, sans-serif;
        }
        table, th, td {
            border: 1px solid black;
            border-collapse: collapse;
        }
        th, td {
            padding: 5px;
            font-family: Helvetica, Arial, sans-serif;
            font-size: 100%;
        }
        tbody tr:nth-child(odd) {background: #eee}
        tbody tr:nth-child(even) {background: #fff}
        table tbody tr td:hover {
            background-color: #fcffb2;
        }
        .row_heading, .blank{
            display: none;
        }
        .col_heading{
            background-color: #CCD1D1;
        }
        .row1, .row3, .row5{
            background-color: #EAEDED;
        }
        .row0:hover, .row2:hover, .row4:hover, .row6:hover{
            background-color: #fcffb2;
        }
        .col1, .col2, .col3, .col4, .col5{
            text-align: right;
        }
        .col0, .col1, .col2, .col3, .col4, .col5{
            font-weight: normal;
        }
        .col_heading, .level0, .blank{
            font-weight: bold;
        }
    </style>
    </head>
    <body>
    """
    
    ss_total = "{:,}".format(int(ss_total))
    tgpl_total = "{:,}".format(int(tgpl_total))
    ss_purchased = "{:,}".format(int(ss_purchased))
    tgpl_purchased = "{:,}".format(int(tgpl_purchased))

    html_str += f"""
    <img src="https://d8tuj5f40nouo.cloudfront.net/images/web/landing/logo.png" width="200" height="83">
    <h2> Hitcoins Usage: {datetime.strftime(end,'%A, %b %d')} </h2>
    <h3>Superstars </h3>
    <h4>Total Hitcoins Spent: {ss_total}</h4> 
    <h4>Hitcoins Purchased - {ss_purchased} </h4>
    {style1}
    <br></br>

    <h3>TGPL </h3>
    <h4>Total Hitcoins Spent: {tgpl_total}</h4> 
    <h4>Hitcoins Purchased - {tgpl_purchased} </h4> 
    {style2}
    </body></html>
    
    """
    subject = f"""Hitcoins: {ss_total} ({ss_purchased})  -  TGPL: {tgpl_total} ({tgpl_purchased}) """

    print('Sending Mail')
    gmail_user = os.environ['mail']
    gmail_password = os.environ['mail_token']
    to = ['digest_hitcoins@hitwicket.com','isha@hitwicket.com']
    sent_from = gmail_user
    
    text = "Please use an html reader"
    message = MIMEMultipart(
        "alternative", None, [MIMEText(text), MIMEText(html_str,'html')])
    
    message['From'] = "Analytics <" + os.environ['mail'] + ">"
    message['To'] = ','.join(to)
    message['Subject'] = subject
    
    s = smtplib.SMTP('smtp.gmail.com', 587) 
    s.starttls() 
    s.login(gmail_user, gmail_password) 
    s.sendmail(sent_from, to, message.as_string())  
    s.quit()
    print('Mail Sent')

except:
   from slacker import Slacker
   slack = Slacker(os.environ['accio'])
   if slack.api.test().successful:
       print( f"Connected to {slack.team.info().body['team']['name']}.")
   else:
       print('Try Again!')
   slack.chat.post_message(channel='cron',
                           text="Cron-job for Hitcoins Mailer on " + str(dt.date.today()) + " has failed.", 
                           username='accio')